# Install Dependencies

In [ ]:
!pip install -q sacrebleu
!pip install -q rouge-score
!pip install -q bert-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 95.6 MB/s eta 0:00:00


In [ ]:
!pip install git+https://github.com/huggingface/evaluate@32546aafec25cdc2a5d7dd9f941fc5be56ba122f

  Cloning https://github.com/huggingface/evaluate (to revision 32546aafec25cdc2a5d7dd9f941fc5be56ba122f) to /tmp/pip-req-build-ipp45rw2
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/evaluate /tmp/pip-req-build-ipp45rw2
  Running command git rev-parse -q --verify 'sha^32546aafec25cdc2a5d7dd9f941fc5be56ba122f'
  Running command git fetch -q https://github.com/huggingface/evaluate 32546aafec25cdc2a5d7dd9f941fc5be56ba122f
  Resolved https://github.com/huggingface/evaluate to commit 32546aafec25cdc2a5d7dd9f941fc5be56ba122f
  Preparing metadata (setup.py) ... done
  Created wheel for evaluate: filename=evaluate-0.4.5.dev0-py3-none-any.whl size=84157 sha256=6d75cbf99093775fdd071eb604b33ffaedee19f6460e26bab31716cb2f627583
  Stored in directory: /root/.cache/pip/wheels/89/96/43/3d812abd89a5bdb5c4e6156d219fdb043db5fb31137b368aee
Successfully built evaluate


# Import Required Modules

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import re

# Optional: Arabic text normalization to improve BLEU alignment
def normalize_arabic(text):
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"[ًٌٍَُِّْ]", "", text)  # Remove short vowels (diacritics)
    text = re.sub(r"[^\w\s]", "", text)    # Remove punctuation (optional)
    return text.strip()

In [ ]:
import huggingface_hub
huggingface_hub.login('HF_TOKEN')

# Zero Shot

In [ ]:
df = pd.read_excel('summarization_data.xlsx')
pred_zero = pd.read_excel('Falcon-TextSummarization-ZeroShot.xlsx')

In [ ]:
pred_zero['summary'] = df['summary']

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import sacrebleu
import evaluate
from transformers import AutoTokenizer

# Initialize storage
bleu_scores = []
bleu_scores2 = []
rougeL_p = []
rougeL_r = []
rougeL_f = []
bertscore_p = []
bertscore_r = []
bertscore_f = []

# Initialize scorer
model_name = 'aubmindlab/bert-base-arabertv2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
rouge = rouge_scorer.RougeScorer(['rougeL'], tokenizer=tokenizer)

# Iterate through rows
for _, row in pred_zero.iterrows():
    raw_references = [row['summary']]  # One reference as list of strings
    raw_predictions = [row['Generated Summary']]  # One prediction

    # Normalize Arabic
    hyp = [normalize_arabic(text) for text in raw_predictions]
    ref = [[normalize_arabic(ref) for ref in raw_references]]  # Nested list for multiple refs per prediction

    # BLEU
    bleu = sacrebleu.corpus_bleu(hyp, ref)
    bleu_scores.append(bleu.score)

    bleu2 = evaluate.load("bleu")
    results = bleu2.compute(predictions= hyp, references= ref)
    bleu_scores2.append(results['bleu'])

    # ROUGE-L
    r_scores = rouge.score(row['summary'], row['Generated Summary'])
    rougeL_p.append(r_scores['rougeL'].precision)
    rougeL_r.append(r_scores['rougeL'].recall)
    rougeL_f.append(r_scores['rougeL'].fmeasure)

# BERTScore
P, R, F = bert_score(pred_zero['Generated Summary'].tolist(), pred_zero['summary'].tolist(), lang="ar", model_type="bert-base-multilingual-cased", verbose=False)
bertscore_p = P.tolist()
bertscore_r = R.tolist()
bertscore_f = F.tolist()

# Create summary DataFrame
metrics_df = pd.DataFrame({
    "BLEU1": bleu_scores,
    "BLEU2": bleu_scores2,
    "ROUGE_L_P": rougeL_p,
    "ROUGE_L_R": rougeL_r,
    "ROUGE_L_F": rougeL_f,
    "BERT_P": bertscore_p,
    "BERT_R": bertscore_r,
    "BERT_F": bertscore_f,
})

# Calculate min, max, mean
summary_stats = metrics_df.agg(['min', 'max', 'mean'])

print(summary_stats)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

          BLEU1     BLEU2  ROUGE_L_P  ROUGE_L_R  ROUGE_L_F    BERT_P  \
min    0.000000  0.000000   0.062500   0.025000   0.036364  0.582208   
max   38.827384  0.388274   0.758621   0.615385   0.539326  0.896750   
mean   4.599717  0.018642   0.288731   0.209402   0.221900  0.738990   

        BERT_R    BERT_F  
min   0.606046  0.615071  
max   0.844575  0.859684  
mean  0.712723  0.724798  


# Fewshot

In [ ]:
df = pd.read_excel('summarization_data.xlsx')
pred_few = pd.read_excel('Falcon-TextSummarization-FewShot.xlsx')

In [ ]:
pred_few['summary'] = df['summary']

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import sacrebleu
import evaluate
from transformers import AutoTokenizer

# Initialize storage
bleu_scores = []
bleu_scores2 = []
rougeL_p = []
rougeL_r = []
rougeL_f = []
bertscore_p = []
bertscore_r = []
bertscore_f = []

# Initialize scorer
model_name = 'aubmindlab/bert-base-arabertv2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
rouge = rouge_scorer.RougeScorer(['rougeL'], tokenizer=tokenizer)

# Iterate through rows
for _, row in pred_few.iterrows():
    raw_references = [row['summary']]  # One reference as list of strings
    raw_predictions = [row['Generated Summary']]  # One prediction

    # Normalize Arabic
    hyp = [normalize_arabic(text) for text in raw_predictions]
    ref = [[normalize_arabic(ref) for ref in raw_references]]  # Nested list for multiple refs per prediction

    # BLEU
    bleu = sacrebleu.corpus_bleu(hyp, ref)
    bleu_scores.append(bleu.score)

    bleu2 = evaluate.load("bleu")
    results = bleu2.compute(predictions= hyp, references= ref)
    bleu_scores2.append(results['bleu'])

    # ROUGE-L
    r_scores = rouge.score(row['summary'], row['Generated Summary'])
    rougeL_p.append(r_scores['rougeL'].precision)
    rougeL_r.append(r_scores['rougeL'].recall)
    rougeL_f.append(r_scores['rougeL'].fmeasure)

# BERTScore
P, R, F = bert_score(pred_few['Generated Summary'].tolist(), pred_few['summary'].tolist(), lang="ar", model_type="bert-base-multilingual-cased", verbose=False)
bertscore_p = P.tolist()
bertscore_r = R.tolist()
bertscore_f = F.tolist()

# Create summary DataFrame
metrics_df = pd.DataFrame({
    "BLEU1": bleu_scores,
    "BLEU2": bleu_scores2,
    "ROUGE_L_P": rougeL_p,
    "ROUGE_L_R": rougeL_r,
    "ROUGE_L_F": rougeL_f,
    "BERT_P": bertscore_p,
    "BERT_R": bertscore_r,
    "BERT_F": bertscore_f,
})

# Calculate min, max, mean
summary_stats = metrics_df.agg(['min', 'max', 'mean'])

print(summary_stats)


          BLEU1     BLEU2  ROUGE_L_P  ROUGE_L_R  ROUGE_L_F    BERT_P  \
min    0.000000  0.000000   0.000000   0.000000   0.000000  0.625462   
max   53.385874  0.533859   1.000000   0.608696   0.666667  0.940987   
mean   4.918981  0.022937   0.327358   0.190526   0.228272  0.751983   

        BERT_R    BERT_F  
min   0.590190  0.609415  
max   0.891093  0.894683  
mean  0.706685  0.728037  


# CoT

In [ ]:
df = pd.read_excel('summarization_data.xlsx')
pred_cot = pd.read_excel('Falcon-TextSummarization-CoT.xlsx')

In [ ]:
pred_cot['summary'] = df['summary']

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import sacrebleu
import evaluate
from transformers import AutoTokenizer

# Initialize storage
bleu_scores = []
bleu_scores2 = []
rougeL_p = []
rougeL_r = []
rougeL_f = []
bertscore_p = []
bertscore_r = []
bertscore_f = []

# Initialize scorer
model_name = 'aubmindlab/bert-base-arabertv2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
rouge = rouge_scorer.RougeScorer(['rougeL'], tokenizer=tokenizer)

# Iterate through rows
for _, row in pred_cot.iterrows():
    raw_references = [row['summary']]  # One reference as list of strings
    raw_predictions = [row['Generated Summary']]  # One prediction

    # Normalize Arabic
    hyp = [normalize_arabic(text) for text in raw_predictions]
    ref = [[normalize_arabic(ref) for ref in raw_references]]  # Nested list for multiple refs per prediction

    # BLEU
    bleu = sacrebleu.corpus_bleu(hyp, ref)
    bleu_scores.append(bleu.score)

    bleu2 = evaluate.load("bleu")
    results = bleu2.compute(predictions= hyp, references= ref)
    bleu_scores2.append(results['bleu'])

    # ROUGE-L
    r_scores = rouge.score(row['summary'], row['Generated Summary'])
    rougeL_p.append(r_scores['rougeL'].precision)
    rougeL_r.append(r_scores['rougeL'].recall)
    rougeL_f.append(r_scores['rougeL'].fmeasure)

# BERTScore
P, R, F = bert_score(pred_cot['Generated Summary'].tolist(), pred_cot['summary'].tolist(), lang="ar", model_type="bert-base-multilingual-cased", verbose=False)
bertscore_p = P.tolist()
bertscore_r = R.tolist()
bertscore_f = F.tolist()

# Create summary DataFrame
metrics_df = pd.DataFrame({
    "BLEU1": bleu_scores,
    "BLEU2": bleu_scores2,
    "ROUGE_L_P": rougeL_p,
    "ROUGE_L_R": rougeL_r,
    "ROUGE_L_F": rougeL_f,
    "BERT_P": bertscore_p,
    "BERT_R": bertscore_r,
    "BERT_F": bertscore_f,
})

# Calculate min, max, mean
summary_stats = metrics_df.agg(['min', 'max', 'mean'])

print(summary_stats)


          BLEU1     BLEU2  ROUGE_L_P  ROUGE_L_R  ROUGE_L_F    BERT_P  \
min    0.000000  0.000000   0.054054   0.033333   0.042553  0.585126   
max   45.340106  0.453401   0.833333   0.606061   0.701754  0.901364   
mean   5.184271  0.024407   0.284852   0.211125   0.225064  0.738487   

        BERT_R    BERT_F  
min   0.605814  0.615005  
max   0.886318  0.886069  
mean  0.713479  0.724962  
